In [1]:
import os, certifi
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from dotenv import load_dotenv

load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")
DB_NAME   = os.getenv("DB_NAME", "Proyecto_Bigdata")

OUTPUT_DIR = os.path.join(os.path.dirname(os.getcwd()), "outputs") if "notebooks" in os.getcwd() else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Leer desde processed_data
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
datos  = list(client[DB_NAME]["processed_data"].find({}, {"_id": 0}))
client.close()

df = pd.DataFrame(datos)
print(f"Registros cargados: {len(df)}")

# Preparar features numéricas
for col_bool in ["es_ti", "es_remoto"]:
    if col_bool not in df.columns:
        df[col_bool] = False
    df[col_bool] = df[col_bool].fillna(False).astype(bool).astype(int)

if "largo_descripcion" not in df.columns:
    df["largo_descripcion"] = df["descripcion"].fillna("").str.len()
df["largo_descripcion"] = pd.to_numeric(df["largo_descripcion"], errors="coerce").fillna(0)

X = df[["largo_descripcion", "es_remoto", "es_ti"]].values

# Escalamiento y PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Varianza explicada por PC1 y PC2: {pca.explained_variance_ratio_.round(3)}")
print(f"Shape PCA: {X_pca.shape}")


Registros cargados: 3719
Varianza explicada por PC1 y PC2: [0.433 0.314]
Shape PCA: (3719, 2)


## Método del Codo — Selección de k óptimo

In [2]:
# Método del Codo — K-Means
inercias = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    print(f"k={k} | Inercia: {km.inertia_:,.2f}")

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), inercias, marker='o', color='steelblue')
plt.title("Método del Codo — K-Means")
plt.xlabel("Número de clústeres (k)")
plt.ylabel("Inercia")
plt.xticks(list(k_range))
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_kmeans_codo.png"), dpi=150)
plt.show()
print("Gráfico del codo guardado")


k=2 | Inercia: 7,175.03
k=3 | Inercia: 4,364.60
k=4 | Inercia: 2,102.62
k=5 | Inercia: 1,395.15
k=6 | Inercia: 837.63
k=7 | Inercia: 609.41
k=8 | Inercia: 418.20
k=9 | Inercia: 263.37
k=10 | Inercia: 189.90
Gráfico del codo guardado


/tmp/ipykernel_12826/1823659014.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## K-Means Final (k=3)

In [3]:
# K-Means Final con k=3
k_optimo = 3
kmeans = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, df["cluster"])
print(f"k={k_optimo} | Inercia: {kmeans.inertia_:,.2f} | Silhouette: {sil:.3f}")

# Perfil de cada clúster
print("\nPerfil de clústeres:")
print(df.groupby("cluster")[["largo_descripcion", "es_remoto", "es_ti"]].mean().round(3))


k=3 | Inercia: 4,364.60 | Silhouette: 0.738

Perfil de clústeres:
         largo_descripcion  es_remoto  es_ti
cluster                                     
0                   96.877      0.313    1.0
1                  115.340      0.000    0.0
2                   94.017      1.000    0.0


In [4]:
# Visualización K-Means en espacio PCA
plt.figure(figsize=(9, 6))
colores = {0: "steelblue", 1: "coral", 2: "seagreen"}
for c in sorted(df["cluster"].unique()):
    mask = df["cluster"] == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colores[c], label=f"Clúster {c}", alpha=0.6, s=30)
plt.title("K-Means (k=3) — Proyección PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_kmeans_pca.png"), dpi=150)
plt.show()
print("Visualización K-Means guardada")


Visualización K-Means guardada


/tmp/ipykernel_12826/3213392428.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## DBSCAN — Detección de clústeres no paramétrica

In [5]:
# DBSCAN — detección de clústeres y outliers
dbscan = DBSCAN(eps=0.8, min_samples=10)
df["cluster_dbscan"] = dbscan.fit_predict(X_scaled)

n_clusters = len(set(df["cluster_dbscan"])) - (1 if -1 in df["cluster_dbscan"].values else 0)
n_ruido    = (df["cluster_dbscan"] == -1).sum()
print(f"DBSCAN — Clústeres encontrados: {n_clusters} | Ruido: {n_ruido} registros")

plt.figure(figsize=(9, 6))
colores_db = {-1: "lightgray"}
palette    = ["steelblue", "coral", "seagreen", "gold", "purple"]
for i, c in enumerate(sorted(set(df["cluster_dbscan"]))):
    if c != -1:
        colores_db[c] = palette[i % len(palette)]

for c, color in colores_db.items():
    mask  = df["cluster_dbscan"] == c
    label = f"Clúster {c}" if c != -1 else "Ruido"
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=color, label=label, alpha=0.6, s=30)
plt.title("DBSCAN — Proyección PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_dbscan_pca.png"), dpi=150)
plt.show()
print("Visualización DBSCAN guardada")


DBSCAN — Clústeres encontrados: 6 | Ruido: 3 registros
Visualización DBSCAN guardada


/tmp/ipykernel_12826/3956656571.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Definición de Variable Dependiente Y — Hito 3

**Variable Y:** `es_ti` (booleano)

Indica si una oferta pertenece al sector TI. Presenta correlaciones con `es_remoto` y `largo_descripcion`. Los clústeres K-Means confirman separación entre ofertas TI y no-TI.

**Algoritmos propuestos para Hito 3:** Regresión Logística y Random Forest.

In [6]:
# Guardar resultados del modelado en JSON para visualización en pipeline
import json as _json

perfil = df.groupby('cluster')[['largo_descripcion','es_remoto','es_ti']].mean().round(3)

resultados_nm = {
    "titulo": "Modelado No Supervisado",
    "algoritmos": ["K-Means", "DBSCAN", "PCA"],
    "kmeans": {
        "k_optimo": 3,
        "inercia": round(float(kmeans.inertia_), 2),
        "silhouette": round(float(sil), 3),
        "perfil_clusters": perfil.to_dict()
    },
    "dbscan": {
        "n_clusters": int(len(set(df['cluster_dbscan'])) - (1 if -1 in df['cluster_dbscan'].values else 0)),
        "n_ruido": int((df['cluster_dbscan'] == -1).sum())
    },
    "variable_Y": {
        "nombre": "es_ti",
        "tipo": "booleano",
        "pct_positivos": round(float(df['es_ti'].mean() * 100), 1),
        "algoritmos_hito3": ["Regresión Logística", "Random Forest"]
    },
    "plots": ["plot_kmeans_codo.png", "plot_kmeans_pca.png", "plot_dbscan_pca.png"]
}

with open(os.path.join(OUTPUT_DIR, 'results_nomodelado.json'), 'w', encoding='utf-8') as f:
    _json.dump(resultados_nm, f, ensure_ascii=False, indent=2, default=str)
print("Resultados NoModelado guardados en outputs/results_nomodelado.json")


Resultados NoModelado guardados en outputs/results_nomodelado.json


In [7]:
## Definición de Variable Dependiente Y — Hito 3

# Variable Y: es_ti (binaria)
print("Distribución de la variable Y (es_ti):")
print(df["es_ti"].value_counts())
print(f"\nProporción TI: {df['es_ti'].mean():.1%}")

# Correlaciones con Y
corr = df[["largo_descripcion", "es_remoto", "es_ti", "cluster"]].corr()
print("\nCorrelaciones con es_ti:")
print(corr["es_ti"].sort_values(ascending=False))


Distribución de la variable Y (es_ti):
es_ti
0    3524
1     195
Name: count, dtype: int64

Proporción TI: 5.2%

Correlaciones con es_ti:
es_ti                1.000000
es_remoto            0.237710
largo_descripcion   -0.074162
cluster             -0.738659
Name: es_ti, dtype: float64
